# Akilli Hat-Arac Atama (Analiz 9 - Synthesis)

**Tarih:** 2026-05-11
**Amac:** Tum onceki analizleri birlestiren karar destek sistemi
**Cikti:** Anomali listesi + operasyonel oneri + FEATURES_FINAL hazirligi

## Yaklasim: Karar Destek Sistemi (Decision Support)
Tam optimization yerine: mevcut atamalardan anomali tespit + degisiklik onerisi + kazanc simulasyonu.

## Test Sistemi
| H | Test | Kabul |
|---|---|---|
| H1 | Mevcut atama optimal degil | hat_zorluk × arac_kritiklik korelasyon > 0 |
| H2 | ARACCINSI uyumsuzluk nadir | < %5 yanlis sinif |
| H3 | Anadolu zor hat + yasli arac (A3 dogrulama) | Garaj × hat_zorluk pozitif |
| H4 | Anomali duzeltme kazanc | Toplam skor %5+ azalma |

## Iki Senaryo Paralel
- **X: Garaj SABIT** (gercekci) - hat icinde rotasyon
- **Y: Garaj SERBEST** (teorik tavan) - garaj degisimi mumkun

## Veri Notu
- ARACTIPI ayrımı ML feature olarak yapılır ama analiz karısık (proje karari)
- ADALAR yok, golf arac yok
- yolcu_doluluk guvenilmez
- ML feature degerlendirmeleri Analiz 8'in stability ders aliyor


---
## 1. Veri Yukleme + Onceki Analizlerden Feature Toplama

Tum onceki analizlerden kanitlanmis feature'lari topluyoruz.


In [1]:
# BOLUM 1: Veri yukleme
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import json as _json

# Ana veri
df = pd.read_csv('../panel_data/temiz_veri/ariza_model.csv', low_memory=False)
df['OLAYTARIHI'] = pd.to_datetime(df['OLAYTARIHI'], format='mixed')
# Bilinmiyor duzeltmesi
mask_b = df['YAKITTURU']=='Bilinmiyor'
df.loc[mask_b & df['MODEL'].str.contains('CNG', na=False), 'YAKITTURU'] = 'CNG'
df.loc[mask_b & ~df['MODEL'].str.contains('CNG', na=False), 'YAKITTURU'] = 'MOTORIN'
df = df[df['YAKITTURU'].isin(['MOTORIN','CNG'])].copy()
print(f'Ariza: {len(df):,}, arac: {df["KAPINO"].nunique():,}')

# Arac × Hat mevcut atama (arac_gunluk_hatlar)
ah = pd.read_csv('../panel_data/temiz_veri/arac_gunluk_hatlar.csv', low_memory=False)
print(f'arac_gunluk_hatlar: {len(ah):,} kayit')
print(f'  Unique arac: {ah["KAPINO"].nunique():,}, unique hat: {ah["HATKODU"].nunique():,}')

# Arac temel profil
arac = df.groupby('KAPINO').agg(
    MARKA=('MARKA','first'),
    MODEL=('MODEL','first'),
    MODELYILI=('MODELYILI','first'),
    ARACCINSI=('ARACCINSI','first'),
    GARAJ=('GARAJ','first'),
    YAKITTURU=('YAKITTURU','first'),
    n_ariza=('ciddi_ariza','count'),
    ort_skor=('ciddiyet_skoru','mean'),
    ciddi_oran=('ciddi_ariza','mean'),
).reset_index()
arac['yas'] = 2025 - arac['MODELYILI']
print(f'\nArac profili: {len(arac):,}')

# Hat temel profil
hat = df.groupby('HATKODU').agg(
    n_ariza=('ciddi_ariza','count'),
    ciddi_oran=('ciddi_ariza','mean'),
    ort_skor=('ciddiyet_skoru','mean'),
    HATCINSI_top=('HATCINSI', lambda s: s.mode()[0] if len(s.mode()) else 'X'),
    hat_uzunluk_ort=('HATUZUNLUK','mean'),
).reset_index()
print(f'Hat profili: {len(hat):,}')
print(hat.head())


Ariza: 58,557, arac: 3,508
arac_gunluk_hatlar: 1,320,647 kayit
  Unique arac: 6,761, unique hat: 836

Arac profili: 3,508
Hat profili: 728
  HATKODU  n_ariza  ciddi_oran  ort_skor HATCINSI_top  hat_uzunluk_ort
0       1       29    0.379310  4.151724         İETT         11456.52
1      10      195    0.389744  3.642513          ÖHO         19029.64
2     10A      227    0.400881  3.762115         İETT         19121.79
3     10B       13    0.384615  3.703846         İETT         13547.87
4     10E       30    0.200000  3.009000         İETT         12026.83


---
## 2. Hat Zorluk Skoru

Her hat icin: egim (Analiz 3) + uzunluk + ariza yogunlugu + sefer trafik
Tum 0-100 normalize, agirlikli toplam.


In [2]:
# BOLUM 2: Hat zorluk skoru
# Egim (Analiz 3)
# Defansif path: kok dizinden veya analiz_9/ subfolder'dan calistirma destegi
import os as _os
_path_cand = ['panel_data/hat_elevation.json', '../panel_data/hat_elevation.json']
_he_path = next((p for p in _path_cand if _os.path.exists(p)), None)
if _he_path is None:
    raise FileNotFoundError(f'hat_elevation.json bulunamadi. Denenen: {_path_cand}')
with open(_he_path, encoding='utf-8') as f:
    he_raw = _json.load(f)
he = pd.DataFrame([
    {'HATKODU': k, 'rakim': v.get('rakım_farkı', 0), 'tirm': v.get('tırmanma_m', 0)}
    for k, v in he_raw.items()
])
def mm_norm(s, q=None):
    if q is not None: s = s.clip(upper=s.quantile(q))
    mn, mx = s.min(), s.max()
    return ((s - mn) / (mx - mn) * 100).round(1) if mx > mn else pd.Series(0.0, index=s.index)
he['norm_r'] = mm_norm(he['rakim'], q=0.99)
he['norm_t'] = mm_norm(he['tirm'], q=0.99)
he['egim_puan'] = (he['norm_r']*0.4 + he['norm_t']*0.6).round(1)
hat = hat.merge(he[['HATKODU','egim_puan']], on='HATKODU', how='left')
hat['egim_puan'] = hat['egim_puan'].fillna(hat['egim_puan'].median())

# Hat sefer trafik (gunluk ortalama)
hat_sefer = ah.groupby('HATKODU').agg(
    sefer_top=('SEFER_SAYISI','sum'),
    n_arac=('KAPINO','nunique'),
    n_gun=('TARIH','nunique'),
).reset_index()
hat_sefer['gunluk_sefer_ort'] = (hat_sefer['sefer_top'] / hat_sefer['n_gun'].clip(lower=1)).round(1)
hat = hat.merge(hat_sefer, on='HATKODU', how='left').fillna(0)

# Normalize ve birlestir
def norm_skor(s):
    if s.max() > s.min():
        return ((s - s.min())/(s.max() - s.min()) * 100).round(1)
    return pd.Series(0.0, index=s.index)
hat['n_egim'] = norm_skor(hat['egim_puan'])
hat['n_uzun'] = norm_skor(hat['hat_uzunluk_ort'])
hat['n_ariza'] = norm_skor(hat['ort_skor'])
hat['n_trafik'] = norm_skor(hat['gunluk_sefer_ort'])

# Hat zorluk skoru (esit agirlik baslangic)
hat['hat_zorluk'] = (hat['n_egim']*0.30 + hat['n_uzun']*0.20 + hat['n_ariza']*0.30 + hat['n_trafik']*0.20).round(1)
print('=== HAT ZORLUK SKORU DAGILIMI ===')
print(hat[['n_egim','n_uzun','n_ariza','n_trafik','hat_zorluk']].describe().round(1))

print('\n=== EN ZOR 15 HAT ===')
print(hat.nlargest(15, 'hat_zorluk')[['HATKODU','HATCINSI_top','egim_puan','hat_uzunluk_ort','ort_skor','gunluk_sefer_ort','hat_zorluk']].to_string(index=False))

print('\n=== EN KOLAY 15 HAT ===')
print(hat.nsmallest(15, 'hat_zorluk')[['HATKODU','HATCINSI_top','egim_puan','hat_uzunluk_ort','ort_skor','gunluk_sefer_ort','hat_zorluk']].to_string(index=False))


=== HAT ZORLUK SKORU DAGILIMI ===
       n_egim  n_uzun  n_ariza  n_trafik  hat_zorluk
count   728.0   728.0    728.0     728.0       728.0
mean     43.9    20.0     31.5       3.8        27.4
std      18.4    11.1     10.6       6.7         8.0
min       0.0     0.0      0.0       0.0         7.0
25%      30.3    12.2     26.7       1.1        21.6
50%      43.7    18.3     31.7       2.2        27.1
75%      54.7    26.1     35.9       4.6        32.6
max     100.0   100.0    100.0     100.0        53.9

=== EN ZOR 15 HAT ===
HATKODU HATCINSI_top  egim_puan  hat_uzunluk_ort  ort_skor  gunluk_sefer_ort  hat_zorluk
   139A          ÖHO       93.0     99427.240000  2.520000              27.9        53.9
   136Z          ÖHO       83.9     21932.010000  7.220000              20.6        53.0
    34G     METROBÜS       66.9     31070.751871  3.761189            1324.6        51.9
   139S         İETT       97.9     56283.540000  3.657609              11.3        50.7
   139D         İETT 

---
## 3. Hat × ARACCINSI Uyumluluk (H2 Testi)

Metrobus hatlari sadece korukluyle yapilmali. Mevcut atamalarda uyumsuzluk var mi?


In [3]:
# BOLUM 3: H2 - ARACCINSI uyumluluk (METROBUS ozeline gore)
ah_arac = ah.merge(arac[["KAPINO","ARACCINSI"]], on="KAPINO", how="left")
hat_arac_cins = ah_arac.merge(hat[["HATKODU","HATCINSI_top"]], on="HATKODU", how="left")

# Metrobus uyumsuzluk (ASIL TEST)
metrobus_mask = hat_arac_cins["HATCINSI_top"]=="METROBÜS"
uyumsuz_metrobus = metrobus_mask & (hat_arac_cins["ARACCINSI"]=="SOLO")
toplam_metrobus = metrobus_mask.sum()
metrobus_uyumsuzluk_pct = uyumsuz_metrobus.sum()/toplam_metrobus*100 if toplam_metrobus > 0 else 0

print("=== H2 TEST: METROBUS UYUMSUZLUK (ASIL TEST) ===")
print(f"Metrobus hatti toplam sefer: {toplam_metrobus:,}")
print(f"  SOLO arac ile yapilan:     {uyumsuz_metrobus.sum():,} (%{metrobus_uyumsuzluk_pct:.4f})")

# Otobus hatlarinda KORUKLU (bilgi, anomali degil)
otobus_mask = hat_arac_cins["HATCINSI_top"].isin(["İETT","KARMA","ÖHO","OAŞ"])
uyumsuz2 = otobus_mask & (hat_arac_cins["ARACCINSI"]=="KORUKLU")
print()
print(f"Otobus hattinda KORUKLU sefer: {uyumsuz2.sum():,} / {otobus_mask.sum():,} (%{uyumsuz2.sum()/otobus_mask.sum()*100:.1f})")
print("  NOT: Bu 'uyumsuzluk' degil - KORUKLU araclar KARMA/OAS hatlarinda DOGAL olarak calisir")

print()
print("=== H2 KARAR (METROBUS ozelinde) ===")
if metrobus_uyumsuzluk_pct < 0.5:
    print(f"  H2 PASSED: Metrobus uyumsuzluk %{metrobus_uyumsuzluk_pct:.4f} (es ihmal edilebilir)")
    print("  Operator metrobus icin KORUKLU sectiriyor - akilli atama")
else:
    print(f"  H2 FAILED: Metrobus uyumsuzluk %{metrobus_uyumsuzluk_pct:.4f} yuksek")

# Toplam uyumsuzluk (bilgi)
toplam_uyumsuzluk_pct = (uyumsuz_metrobus.sum() + uyumsuz2.sum()) / len(hat_arac_cins) * 100
print()
print(f"[Bilgi] Toplam karisik metrik: %{toplam_uyumsuzluk_pct:.2f} (yaniltici - KORUKLU otobus hatlarinda dogal)")

print()
print("=== HAT_CINSI x ARACCINSI CAPRAZ (sefer sayisi) ===")
ct = pd.crosstab(hat_arac_cins["HATCINSI_top"], hat_arac_cins["ARACCINSI"])
print(ct)


=== H2 TEST: METROBUS UYUMSUZLUK (ASIL TEST) ===
Metrobus hatti toplam sefer: 321,256
  SOLO arac ile yapilan:     15 (%0.0047)

Otobus hattinda KORUKLU sefer: 77,821 / 843,222 (%9.2)
  NOT: Bu 'uyumsuzluk' degil - KORUKLU araclar KARMA/OAS hatlarinda DOGAL olarak calisir

=== H2 KARAR (METROBUS ozelinde) ===
  H2 PASSED: Metrobus uyumsuzluk %0.0047 (es ihmal edilebilir)
  Operator metrobus icin KORUKLU sectiriyor - akilli atama

[Bilgi] Toplam karisik metrik: %5.89 (yaniltici - KORUKLU otobus hatlarinda dogal)

=== HAT_CINSI x ARACCINSI CAPRAZ (sefer sayisi) ===
ARACCINSI     KORUKLU    SOLO
HATCINSI_top                 
ELEKTRİKLİ          1     197
KARMA           19618   79782
METROBÜS       321239      15
OAŞ              1457   10374
TAHSİS             16    1705
ÖHO              4241   23279
ÖÇK                 2      44
İETT            52505  248409


---
## 4. Arac Uygunluk Skoru

Her arac icin: kritiklik (A8) + verimsizlik (A7) + garaj_lift (A5) + yas + gecmis_ciddi_oran
Yuksek skor = riskli, düsük skor = guvenli.


In [4]:
# BOLUM 4: Arac uygunluk skoru
# Onceki analizlerden feature'lar (yeniden hesap)

# verimsizlik_skoru (A7)
tuketim = pd.DataFrame([
    ('OTOKAR','KENT 290LF',40),('OTOKAR','KENT XL',60),
    ('MERCEDES','CITARO 0530',39),('MERCEDES','CITARO 0530 G',58),
    ('MERCEDES','CONECTO G',62),('MERCEDES','CONECTO',42),
    ('MERCEDES','CAPACITY',65),
    ('BMC','PROCITY TR',41),('BMC','PROCITY',41),
    ('KARSAN','AVANCITY S PLUS',58),('KARSAN','AVANCITY CNG',52),
    ('TEMSA','AVENUE LF CNG',50),
    ('AKIA','ULTRA LF12',40),('AKIA','LF25',60),
], columns=['MARKA','MODEL','tuketim_100km'])
arac = arac.merge(tuketim, on=['MARKA','MODEL'], how='left')
arac['tuketim_100km'] = arac['tuketim_100km'].fillna(arac['tuketim_100km'].median())
arac['yas_norm'] = norm_skor(arac['yas'])
arac['tuketim_norm'] = norm_skor(arac['tuketim_100km'])
arac['verimsizlik_skoru'] = (arac['yas_norm'] * arac['tuketim_norm'] / 100).round(2)

# garaj_ort_skor (A5)
garaj_ort = df.groupby('GARAJ')['ciddiyet_skoru'].mean().reset_index()
garaj_ort.columns = ['GARAJ','garaj_ort_skor']
arac = arac.merge(garaj_ort, on='GARAJ', how='left')

# Tum normalize
arac['n_yas'] = norm_skor(arac['yas'])
arac['n_ariza'] = norm_skor(arac['ort_skor'])
arac['n_verims'] = norm_skor(arac['verimsizlik_skoru'])
arac['n_garaj'] = norm_skor(arac['garaj_ort_skor'])

# Arac uygunluk skoru: r-orantili agirlik (Analiz 8'den)
# garaj_ort_skor en guclu (r=0.375), digerler benzer
arac['arac_kritiklik'] = (
    arac['n_garaj']*0.35 +    # Analiz 5/8 dominant
    arac['n_ariza']*0.20 +    # gecmis performans
    arac['n_verims']*0.15 +   # Analiz 7
    arac['n_yas']*0.15 +      # motor yenileme kisitiyla
    arac['ciddi_oran']*100*0.15  # mevcut ciddi oran
).round(1)
print('=== ARAC KRITIKLIK SKORU ===')
print(arac['arac_kritiklik'].describe().round(1))
print(f'\nEn riskli 10 arac:')
print(arac.nlargest(10, 'arac_kritiklik')[['KAPINO','GARAJ','MARKA','yas','arac_kritiklik']].to_string(index=False))
print(f'\nEn guvenli 10 arac:')
print(arac.nsmallest(10, 'arac_kritiklik')[['KAPINO','GARAJ','MARKA','yas','arac_kritiklik']].to_string(index=False))


=== ARAC KRITIKLIK SKORU ===
count    3508.0
mean       50.7
std        13.4
min         0.0
25%        44.6
50%        51.5
75%        58.3
max        91.2
Name: arac_kritiklik, dtype: float64

En riskli 10 arac:
KAPINO      GARAJ    MARKA  yas  arac_kritiklik
 M3149 Edirnekapı MERCEDES 17.0            91.2
 M6509  Şahinkaya MERCEDES 19.0            85.7
 M2988  Şahinkaya MERCEDES 19.0            85.6
 M3100  Şahinkaya MERCEDES 19.0            84.4
 M2038  Şahinkaya MERCEDES 19.0            84.0
 M6512  Şahinkaya MERCEDES 19.0            83.3
 M4597  Şahinkaya MERCEDES 19.0            83.2
 M3097  Şahinkaya MERCEDES 19.0            83.0
 M2985  Şahinkaya MERCEDES 19.0            81.0
 M4358  Şahinkaya MERCEDES 19.0            81.0

En guvenli 10 arac:
KAPINO   GARAJ MARKA  yas  arac_kritiklik
 A9463 Topkapı  AKIA  1.0             0.0
 A9518 Topkapı  AKIA  1.0             0.0
 A9538 Topkapı  AKIA  1.0             0.0
 A9594 Topkapı  AKIA  1.0             0.0
 A9596 Topkapı  AKIA  1.0  

---
## 5. Mevcut Hat × Arac Atama Matrisi

Hangi arac hangi hatta ne kadar siklikla calistirilmis?


In [5]:
# BOLUM 5: Mevcut atama
atama = ah.groupby(['KAPINO','HATKODU'])['SEFER_SAYISI'].sum().reset_index()
atama.columns = ['KAPINO','HATKODU','toplam_sefer']
print(f'Atama kayit: {len(atama):,}')

# Her arac icin ana hat (en cok sefer yaptigi)
arac_ana_hat = atama.sort_values('toplam_sefer', ascending=False).groupby('KAPINO').head(1)
arac_ana_hat.columns = ['KAPINO','ana_HAT','ana_HAT_sefer']
arac = arac.merge(arac_ana_hat, on='KAPINO', how='left')
print(f'\n=== ARAC ANA HAT ATAMASI ===')
print(arac[['KAPINO','GARAJ','ana_HAT','ana_HAT_sefer']].head(10).to_string(index=False))

# Her hat icin ana arac sayisi
hat_arac_sayi = atama.groupby('HATKODU')['KAPINO'].nunique().reset_index(name='kullanilan_arac_sayi')
hat = hat.merge(hat_arac_sayi, on='HATKODU', how='left').fillna(0)
print(f'\n=== HAT BASINA KULLANILAN ARAC SAYISI ===')
print(hat['kullanilan_arac_sayi'].describe().round(1))


Atama kayit: 154,584

=== ARAC ANA HAT ATAMASI ===
KAPINO      GARAJ ana_HAT  ana_HAT_sefer
 A3400 Edirnekapı    34AS            456
 A3401 Edirnekapı    34AS            334
 A3402 Edirnekapı    34AS            181
 A3403 Edirnekapı    34AS            410
 A3404 Edirnekapı    34AS            404
 A3405 Edirnekapı    34AS            469
 A3406 Edirnekapı    34AS            380
 A3407 Edirnekapı    34BZ            418
 A3416 Edirnekapı    34BZ            407
 A3417 Edirnekapı    34AS            438

=== HAT BASINA KULLANILAN ARAC SAYISI ===
count    728.0
mean     202.9
std      149.5
min        0.0
25%       93.8
50%      162.0
75%      258.2
max      831.0
Name: kullanilan_arac_sayi, dtype: float64


---
## 6. H1 Testi: Mevcut Atama Optimal mi?

Zor hatlara riskli araclar atanmis mi (yanlis eslesme) yoksa operator zaten akilli mi?


In [6]:
# BOLUM 6: H1 - mevcut atama optimal mi?
# Her atama kaydı icin: hat_zorluk × arac_kritiklik
atama_skor = atama.merge(hat[['HATKODU','hat_zorluk']], on='HATKODU', how='left')
atama_skor = atama_skor.merge(arac[['KAPINO','arac_kritiklik','GARAJ','MARKA','yas']], on='KAPINO', how='left')
atama_skor['eslesme_skoru'] = (atama_skor['hat_zorluk'] * atama_skor['arac_kritiklik'] / 100).round(2)
# Agirlikli ortalama: sefer sayisi ile
atama_skor['agirlikli_skor'] = atama_skor['eslesme_skoru'] * atama_skor['toplam_sefer']

# Korelasyon: hat_zorluk × arac_kritiklik
r_h1, p_h1 = stats.pearsonr(atama_skor['hat_zorluk'].fillna(0), atama_skor['arac_kritiklik'].fillna(0))
print(f'=== H1 TEST: hat_zorluk × arac_kritiklik korelasyonu ===')
print(f'Pearson r = {r_h1:+.4f}, p = {p_h1:.6e}')
if r_h1 > 0.05:
    print(f'  H1 PASSED: Zor hat × riskli arac eslestirmesi var (r>0.05)')
    print(f'  Yorum: Operator zor hatlara DAHA AZ guvenilir araclar atiyor (anomali)')
elif r_h1 < -0.05:
    print(f'  H1 FAILED: Operator akilli - zor hata guvenli arac atiyor (negatif korelasyon)')
else:
    print(f'  H1 NEUTRAL: Korelasyon zayif - atama hat zorlugundan bagimsiz')

# Hat zorluk bantlari × arac kritiklik
print(f'\n=== HAT ZORLUK BANT × ORT ARAC KRITIKLIK ===')
atama_skor['hat_zor_bant'] = pd.qcut(atama_skor['hat_zorluk'], q=4, labels=['Q1_kolay','Q2','Q3','Q4_zor'], duplicates='drop')
bant = atama_skor.groupby('hat_zor_bant', observed=True).agg(
    n_atama=('KAPINO','count'),
    ort_arac_kritik=('arac_kritiklik','mean'),
    ort_hat_zorluk=('hat_zorluk','mean'),
).round(2)
print(bant.to_string())


=== H1 TEST: hat_zorluk × arac_kritiklik korelasyonu ===
Pearson r = +0.3384, p = 0.000000e+00
  H1 PASSED: Zor hat × riskli arac eslestirmesi var (r>0.05)
  Yorum: Operator zor hatlara DAHA AZ guvenilir araclar atiyor (anomali)

=== HAT ZORLUK BANT × ORT ARAC KRITIKLIK ===
              n_atama  ort_arac_kritik  ort_hat_zorluk
hat_zor_bant                                          
Q1_kolay        38110            43.36           19.53
Q2              37015            47.16           26.15
Q3              36586            48.69           31.34
Q4_zor          36016            49.90           39.11


---
## 6.5. H1 Testi — Garaj-İçi Saflaştırılmış Korelasyon (B1 — FWL Doğrulaması)

**Soru:** Bölüm 6'daki r=+0.338 anomali sinyali **gerçek bir atama hatası mı**, yoksa **garaj-coğrafya confounder'ının** istatistiksel yansıması mı?

**Yöntem:** Frisch-Waugh-Lovell teoremi. GARAJ etkisini hem `arac_kritiklik`'ten hem `hat_zorluk`'tan **residualize** edip artıkların korelasyonunu hesapla. Bu, garajı sabit tutan kısmi korelasyona (partial r) eşdeğer.

**Beklenti:**
- Eğer FWL r ≪ 0.338 → orijinal sinyal büyük ölçüde garaj-coğrafya kaynaklı (mevcut A9 SONUCLAR yorumu doğru)
- Eğer FWL r ≈ 0.10+ ise hâlâ → garaj-içi anomali var, A3'ün "politika ters" tezi güçleniyor → A9 SONUCLAR'a revize yorum eklenmeli

In [7]:
# BOLUM 6.5: H1 Testi — Garaj-Ici Saf Korelasyon (Frisch-Waugh-Lovell)
# B1 — V6.5 etkilemez, sadece bilgilendirici test.

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from statsmodels.formula.api import ols

# Defansif: atama_skor (Bolum 6 dan) var mi?
if "atama_skor" not in globals():
    raise RuntimeError("atama_skor bulunamadi. Once Bolum 6 hucresini calistirin.")

# NaN temizligi (defansif)
_df = atama_skor[["hat_zorluk", "arac_kritiklik", "GARAJ"]].dropna().copy()
print(f"FWL veri seti: {len(_df):,} atama kaydi, {_df['GARAJ'].nunique()} garaj")
print()

# 1) arac_kritiklik ~ C(GARAJ) -> residual
m_arac = ols("arac_kritiklik ~ C(GARAJ)", data=_df).fit()
_df["resid_arac"] = m_arac.resid
print("Adim 1: arac_kritiklik ~ C(GARAJ) regresyonu")
print(f"  R^2 (garaj acikladigi arac_kritiklik varyansi): {m_arac.rsquared:.4f}")
print(f"  Yani arac_kritiklik varyansinin %{m_arac.rsquared*100:.1f} kismi GARAJ ile aciklanabiliyor.")
print()

# 2) hat_zorluk ~ C(GARAJ) -> residual
m_hat = ols("hat_zorluk ~ C(GARAJ)", data=_df).fit()
_df["resid_hat"] = m_hat.resid
print("Adim 2: hat_zorluk ~ C(GARAJ) regresyonu")
print(f"  R^2 (garaj acikladigi hat_zorluk varyansi): {m_hat.rsquared:.4f}")
print(f"  Yani hat_zorluk varyansinin %{m_hat.rsquared*100:.1f} kismi GARAJ (cografi) sabitiyle aciklanabiliyor.")
print()

# 3) Artiklarin korelasyonu (FWL = garaj-ici kismi korelasyon)
r_saf, p_saf = pearsonr(_df["resid_arac"], _df["resid_hat"])

# Kiyaslama icin orijinal
r_ham, p_ham = pearsonr(_df["arac_kritiklik"], _df["hat_zorluk"])

print("=" * 70)
print("SONUC — H1 testi: Yuzeysel vs Garaj-Ici Saf Korelasyon")
print("=" * 70)
print(f'{"Test":<45s} {"r":>10s} {"p":>10s}')
print("-" * 70)
print(f'{"Bolum 6 H1 (ham)":<45s} {r_ham:+10.4f} {p_ham:>10.4f}')
print(f'{"FWL (garaj sabit, kismi korelasyon)":<45s} {r_saf:+10.4f} {p_saf:>10.4f}')
print("-" * 70)

dusus_pct = (1 - abs(r_saf) / abs(r_ham)) * 100 if abs(r_ham) > 0.001 else 0
print(f"Garaj confounder etkisi: r {r_ham:+.4f} -> {r_saf:+.4f}, dusus %{dusus_pct:.1f}")
print()

# Karar agaci
print("YORUM:")
if abs(r_saf) < 0.05:
    print("  >> SIFIRA YAKIN: Garaj sabit tutuldugunda anomali pratik olarak yok oluyor.")
    print("  >> Bolum 6 r=+0.338 sinyali TAMAMEN garaj-cografya confounder kaynakli.")
    print("  >> A9 SONUCLAR raporundaki kismen tautolojik yorumu DOGRULANIYOR.")
    print("  >> Atama hatasi yok; mevcut atama, garaj kisiti altinda zaten optimal.")
elif abs(r_saf) < 0.15:
    print("  >> ZAYIF AMA POZITIF: Garaj sabit tutuldugunda anomali kuculuyor ama tamamen yok olmuyor.")
    print("  >> Sinyal cogunlukla confounder, ama %15-30 araliginda garaj-ici saf anomali da var.")
    print("  >> A9 SONUCLAR mevcut hali ile dogru cerceveliyor, ek nuans eklenebilir.")
else:
    print("  >> POZITIF VE GUCLU: Garaj-ici bile anomali korunuyor.")
    print("  >> A3 notebook raporundaki politika ters tezi guclenir.")
    print("  >> A9 SONUCLAR H1 yorumu REVIZE edilmeli — kismen tautolojik yerine")
    print("     garaj-icinde de yasli arac zor hatta denmeli.")

# Anlamlilik
print()
if p_saf < 0.001:
    print(f"  Istatistiksel anlamlilik: p={p_saf:.2e} - cok yuksek anlamli (buyuk n nedeniyle)")
elif p_saf < 0.05:
    print(f"  Istatistiksel anlamlilik: p={p_saf:.4f} - anlamli")
else:
    print(f"  Istatistiksel anlamlilik: p={p_saf:.4f} - anlamsiz")


FWL veri seti: 132,149 atama kaydi, 12 garaj

Adim 1: arac_kritiklik ~ C(GARAJ) regresyonu
  R^2 (garaj acikladigi arac_kritiklik varyansi): 0.8464
  Yani arac_kritiklik varyansinin %84.6 kismi GARAJ ile aciklanabiliyor.

Adim 2: hat_zorluk ~ C(GARAJ) regresyonu
  R^2 (garaj acikladigi hat_zorluk varyansi): 0.1275
  Yani hat_zorluk varyansinin %12.8 kismi GARAJ (cografi) sabitiyle aciklanabiliyor.

SONUC — H1 testi: Yuzeysel vs Garaj-Ici Saf Korelasyon
Test                                                   r          p
----------------------------------------------------------------------
Bolum 6 H1 (ham)                                 +0.2150     0.0000
FWL (garaj sabit, kismi korelasyon)              +0.0508     0.0000
----------------------------------------------------------------------
Garaj confounder etkisi: r +0.2150 -> +0.0508, dusus %76.4

YORUM:
  >> ZAYIF AMA POZITIF: Garaj sabit tutuldugunda anomali kuculuyor ama tamamen yok olmuyor.
  >> Sinyal cogunlukla confounder, ama

---
## 7. H3 Testi: Garaj × Hat Zorluk (Analiz 3 Dogrulama)

Anadolu zor hatlarda yasli arac mi? Topkapi kolay hatlarda yeni arac mi?


In [8]:
# BOLUM 7: H3 - garaj × hat zorluk
garaj_hat = atama_skor.groupby('GARAJ').agg(
    n_atama=('KAPINO','count'),
    ort_hat_zorluk=('hat_zorluk','mean'),
    ort_arac_kritik=('arac_kritiklik','mean'),
    ort_yas=('yas','mean'),
).round(2).sort_values('ort_hat_zorluk', ascending=False)
print('=== GARAJ × ORT HAT ZORLUK ===')
print(garaj_hat.to_string())

# H3 testi: yas × hat_zorluk korelasyonu (garaj bazli)
r_h3, p_h3 = stats.pearsonr(garaj_hat['ort_yas'], garaj_hat['ort_hat_zorluk'])
print(f'\nGaraj yas × hat_zorluk korelasyon: r={r_h3:+.4f}, p={p_h3:.4f}')
if r_h3 > 0.3:
    print('  H3 PASSED: Yasli filolu garajlar daha zor hatlarda (Analiz 3 dogrulandi)')
elif r_h3 < -0.3:
    print('  H3 INVERSE: Yasli garajlar kolay hatlarda (beklenmedik)')
else:
    print('  H3 NEUTRAL: Iliski zayif')

# Anadolu vs Topkapi spesifik
print(f'\n=== ANADOLU vs TOPKAPI (Analiz 3 spesifik) ===')
if 'Anadolu' in garaj_hat.index and 'Topkapı' in garaj_hat.index:
    a = garaj_hat.loc['Anadolu']
    t = garaj_hat.loc['Topkapı']
    print(f'Anadolu:  yas={a["ort_yas"]:.1f}, hat_zorluk={a["ort_hat_zorluk"]:.1f}, kritik={a["ort_arac_kritik"]:.1f}')
    print(f'Topkapı: yas={t["ort_yas"]:.1f}, hat_zorluk={t["ort_hat_zorluk"]:.1f}, kritik={t["ort_arac_kritik"]:.1f}')


=== GARAJ × ORT HAT ZORLUK ===
                           n_atama  ort_hat_zorluk  ort_arac_kritik  ort_yas
GARAJ                                                                       
Şahinkaya                     3280           36.44            65.76    19.00
Edirnekapı                    2781           35.53            61.11    12.03
Hasanpaşa                     2388           35.27            54.44     8.03
KURTKÖY                      16461           30.82            55.88    12.08
IKITELLIISLETTIRMEGARAJI2    13510           29.50            49.33    11.07
Sarıgazi                      8659           29.17            49.21    11.99
IKITELLIGARAJI               27322           29.10            42.76     8.35
Anadolu                      15743           29.04            53.43    18.21
SULTANGAZIGARAJI             16795           28.94            46.47    12.09
Yunus                         8793           28.45            42.91    12.00
Kağıthane                     9640           

---
## 8. Anomali Tespit - Top 50 Yanlıs Eslesme

Yuksek hat_zorluk × yuksek arac_kritiklik kombinasyonlari = en yanlis eslesmeler.


In [9]:
# BOLUM 8: Anomali tespit
# Sadece ana hat atamalarini al (her arac icin)
arac_ana = arac.merge(hat[['HATKODU','hat_zorluk','HATCINSI_top']],
                      left_on='ana_HAT', right_on='HATKODU', how='left')
arac_ana['anomali_skor'] = (arac_ana['hat_zorluk'] * arac_ana['arac_kritiklik'] / 100).round(1)

print('=== TOP 50 ANOMALI (yuksek skor = en yanlis eslesme) ===')
top50_anomali = arac_ana.nlargest(50, 'anomali_skor')[
    ['KAPINO','GARAJ','MARKA','MODEL','yas','arac_kritiklik','ana_HAT','HATCINSI_top','hat_zorluk','anomali_skor']
]
print(top50_anomali.to_string(index=False))

print(f'\n=== TOP 50 GARAJ DAGILIMI ===')
print(top50_anomali['GARAJ'].value_counts())

print(f'\n=== TOP 50 HAT DAGILIMI ===')
print(top50_anomali['ana_HAT'].value_counts().head(10))


=== TOP 50 ANOMALI (yuksek skor = en yanlis eslesme) ===
KAPINO      GARAJ    MARKA       MODEL  yas  arac_kritiklik ana_HAT HATCINSI_top  hat_zorluk  anomali_skor
 M4003 Edirnekapı MERCEDES    CAPACITY 18.0            78.8    34BZ     METROBÜS        49.4          38.9
 M3292 Edirnekapı MERCEDES    CAPACITY 16.0            78.0    34BZ     METROBÜS        49.4          38.5
 M4544 Edirnekapı MERCEDES    CAPACITY 17.0            76.9    34BZ     METROBÜS        49.4          38.0
 M6576 Edirnekapı MERCEDES    CAPACITY 18.0            76.3    34BZ     METROBÜS        49.4          37.7
 M6591 Edirnekapı MERCEDES    CAPACITY 17.0            75.2    34BZ     METROBÜS        49.4          37.1
 M3286 Edirnekapı MERCEDES    CAPACITY 16.0            74.6    34BZ     METROBÜS        49.4          36.9
 M4411 Edirnekapı MERCEDES    CAPACITY 18.0            74.7    34BZ     METROBÜS        49.4          36.9
 M6585 Edirnekapı MERCEDES    CAPACITY 17.0            74.2    34BZ     METROBÜS       

---
## 9. Anomali Profili - Yas ve Modele Gore Dagilim

Anomali araclar kim? Yasli mi yeni mi? Hangi modeller?


In [10]:
# BOLUM 9: Anomali profili
# Tum araclar icin anomali skoru
print('=== ANOMALI SKOR DAGILIMI ===')
print(arac_ana['anomali_skor'].describe().round(1))

print(f'\n=== ANOMALI BANTLARI × ORT YAS, ORT KRITIK ===')
arac_ana['anomali_bant'] = pd.qcut(arac_ana['anomali_skor'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')
prof = arac_ana.groupby('anomali_bant', observed=True).agg(
    n=('KAPINO','count'),
    ort_yas=('yas','mean'),
    ort_kritik=('arac_kritiklik','mean'),
    ort_hat_zorluk=('hat_zorluk','mean'),
).round(2)
print(prof.to_string())

# Top 50 model dagilimi
print(f'\n=== TOP 50 ANOMALI MODEL DAGILIMI ===')
top50_anomali = arac_ana.nlargest(50, 'anomali_skor')
print(top50_anomali.groupby(['MARKA','MODEL']).size().sort_values(ascending=False))


=== ANOMALI SKOR DAGILIMI ===
count    3508.0
mean       17.4
std         8.0
min         0.0
25%        12.0
50%        16.1
75%        22.2
max        38.9
Name: anomali_skor, dtype: float64

=== ANOMALI BANTLARI × ORT YAS, ORT KRITIK ===
                n  ort_yas  ort_kritik  ort_hat_zorluk
anomali_bant                                          
Q1            712     8.94       36.66           19.52
Q2            707    12.24       48.82           26.84
Q3            691    12.41       50.68           32.54
Q4            703    10.49       53.56           39.80
Q5            695    13.80       63.90           46.99

=== TOP 50 ANOMALI MODEL DAGILIMI ===
MARKA     MODEL      
MERCEDES  CAPACITY       48
          CITARO 0530     2
dtype: int64


---
## 10. Senaryo X: GARAJ SABIT - Hat Icinde Rotasyon

Gercekci senaryo: arac garajini degistiremez ama ayni garaj icindeki diger araclarla yer degistirebilir.


In [11]:
# BOLUM 10: Senaryo X - Garaj Sabit
# Her garaj icin: en riskli araci en kolay hata, en guvenli araci en zor hata yonlendir
def senaryo_garaj_sabit(arac_df, atama_df, hat_df):
    onerileri = []
    for garaj, sub_arac in arac_df.groupby('GARAJ'):
        # Bu garajin araclari ve sirali kritiklik
        sub_sirali = sub_arac.sort_values('arac_kritiklik', ascending=False).reset_index(drop=True)
        # Bu garaj araclarinin atamali hatlarini al
        hatlar_garaj = atama_df[atama_df['KAPINO'].isin(sub_arac['KAPINO'])]
        hatlar_sirali = hatlar_garaj.merge(hat_df[['HATKODU','hat_zorluk']], on='HATKODU', how='left')
        hatlar_sirali = hatlar_sirali.sort_values('hat_zorluk', ascending=True)
        # Eslesme: en kritik arac -> en kolay hat, en guvenli -> en zor
        for i in range(min(len(sub_sirali), len(hatlar_sirali))):
            arac_k = sub_sirali.iloc[i]
            hat_k = hatlar_sirali.iloc[-(i+1) if i < len(hatlar_sirali)/2 else i]
            onerileri.append({
                'KAPINO': arac_k['KAPINO'],
                'GARAJ': garaj,
                'arac_kritiklik': arac_k['arac_kritiklik'],
                'mevcut_hat': arac_k.get('ana_HAT'),
                'onerilen_hat': hat_k['HATKODU'],
                'mevcut_hat_zorluk': arac_ana[arac_ana['KAPINO']==arac_k['KAPINO']]['hat_zorluk'].iloc[0] if (arac_ana['KAPINO']==arac_k['KAPINO']).any() else 0,
                'onerilen_hat_zorluk': hat_k['hat_zorluk'],
            })
    return pd.DataFrame(onerileri)

print('=== SENARYO X: GARAJ SABIT ===')
senaryo_x = senaryo_garaj_sabit(arac, atama, hat)
print(f'Toplam oneri: {len(senaryo_x):,}')

# Mevcut vs onerilen toplam eslesme skoru
mevcut_toplam = (arac_ana['arac_kritiklik'] * arac_ana['hat_zorluk'] / 100).sum()
senaryo_x['onerilen_eslesme'] = (senaryo_x['arac_kritiklik'] * senaryo_x['onerilen_hat_zorluk'] / 100)
onerilen_toplam = senaryo_x['onerilen_eslesme'].sum()

kazanc = (mevcut_toplam - onerilen_toplam) / mevcut_toplam * 100
print(f'\nMevcut toplam eslesme skoru: {mevcut_toplam:,.0f}')
print(f'Onerilen toplam (garaj sabit):  {onerilen_toplam:,.0f}')
print(f'KAZANC: %{kazanc:.2f}')


=== SENARYO X: GARAJ SABIT ===
Toplam oneri: 3,508

Mevcut toplam eslesme skoru: 61,073
Onerilen toplam (garaj sabit):  60,716
KAZANC: %0.58


---
## 11. Senaryo Y: GARAJ SERBEST + ASIMETRIK ARACCINSI KISITI

Gercek fiziksel kisit asimetrik:
- METROBUS hatlari → SADECE KORUKLU (fiziksel zorunluluk)
- Otobus hatlari → HEM KORUKLU HEM SOLO (esnek)
- SOLO arac → sadece otobus hatlarina gidebilir
- KORUKLU arac → her iki turde de calisabilir

Mantik: Metrobus hatlari yogun ve zor (1300-1700 sefer/gun). Onlara en GUVENILIR (kritikligi dusuk) KORUKLU araclari ata. Kalan KORUKLU + tum SOLO → otobus hatlarina greedy kritiklik bazli.


In [12]:
# BOLUM 11: Senaryo Y - ASIMETRIK ARACCINSI KISITI
# METROBUS hatlari -> sadece KORUKLU
# Otobus hatlari -> hem KORUKLU hem SOLO
# SOLO arac -> sadece otobus hatlarina

# Hat kategorileri
hat["ARACCINSI_gerek"] = hat["HATCINSI_top"].apply(
    lambda h: "KORUKLU" if h == "METROBÜS" else "SERBEST"
)
print("Hat ARACCINSI gereksinim dagilimi:")
print(hat["ARACCINSI_gerek"].value_counts())
print()

# 1. Metrobus hatlari icin: en GUVENILIR (en dusuk kritiklik) KORUKLU araclari ata
metrobus_hat = hat[hat["ARACCINSI_gerek"]=="KORUKLU"].sort_values("hat_zorluk", ascending=False).reset_index(drop=True)
print(f"METROBUS hat sayisi: {len(metrobus_hat)}")

koruklu_arac = arac[arac["ARACCINSI"]=="KORUKLU"].copy().sort_values("arac_kritiklik", ascending=True).reset_index(drop=True)
print(f"KORUKLU arac toplam: {len(koruklu_arac)}")

n_metrobus = len(metrobus_hat)
koruklu_metrobus = koruklu_arac.head(n_metrobus).copy()
koruklu_kalan = koruklu_arac.iloc[n_metrobus:].copy()

print(f"Metrobus icin tahsis edilen guvenilir KORUKLU: {len(koruklu_metrobus)}")
print(f"Otobus hatlarina kalan KORUKLU: {len(koruklu_kalan)}")

metrobus_eslesme = pd.DataFrame({
    "KAPINO": koruklu_metrobus["KAPINO"].values,
    "arac_kritiklik": koruklu_metrobus["arac_kritiklik"].values,
    "onerilen_hat": metrobus_hat["HATKODU"].values,
    "onerilen_hat_zorluk": metrobus_hat["hat_zorluk"].values,
    "kategori": "METROBUS",
})
metrobus_eslesme["onerilen_eslesme"] = metrobus_eslesme["arac_kritiklik"] * metrobus_eslesme["onerilen_hat_zorluk"] / 100

# 2. Otobus hatlari: KORUKLU_kalan + SOLO arac havuzu
solo_arac = arac[arac["ARACCINSI"]=="SOLO"].copy()
otobus_havuz = pd.concat([koruklu_kalan, solo_arac], ignore_index=True).sort_values("arac_kritiklik", ascending=False).reset_index(drop=True)
otobus_hat = hat[hat["ARACCINSI_gerek"]=="SERBEST"].sort_values("hat_zorluk", ascending=True).reset_index(drop=True)
print()
print(f"Otobus havuzu (KORUKLU_kalan + SOLO): {len(otobus_havuz)} arac")
print(f"Otobus hat sayisi: {len(otobus_hat)}")

n_min = min(len(otobus_havuz), len(otobus_hat))
otobus_eslesme = pd.DataFrame({
    "KAPINO": otobus_havuz.head(n_min)["KAPINO"].values,
    "arac_kritiklik": otobus_havuz.head(n_min)["arac_kritiklik"].values,
    "onerilen_hat": otobus_hat.head(n_min)["HATKODU"].values,
    "onerilen_hat_zorluk": otobus_hat.head(n_min)["hat_zorluk"].values,
    "kategori": "OTOBUS",
})
remain = otobus_havuz.iloc[n_min:].copy()
if len(remain) > 0:
    np.random.seed(42)
    sample = otobus_hat.sample(len(remain), replace=True, random_state=42)
    extra = pd.DataFrame({
        "KAPINO": remain["KAPINO"].values,
        "arac_kritiklik": remain["arac_kritiklik"].values,
        "onerilen_hat": sample["HATKODU"].values,
        "onerilen_hat_zorluk": sample["hat_zorluk"].values,
        "kategori": "OTOBUS_extra",
    })
    otobus_eslesme = pd.concat([otobus_eslesme, extra], ignore_index=True)

otobus_eslesme["onerilen_eslesme"] = otobus_eslesme["arac_kritiklik"] * otobus_eslesme["onerilen_hat_zorluk"] / 100

senaryo_y_v3 = pd.concat([metrobus_eslesme, otobus_eslesme], ignore_index=True)
onerilen_y_v3 = senaryo_y_v3["onerilen_eslesme"].sum()
kazanc_y_v3 = (mevcut_toplam - onerilen_y_v3) / mevcut_toplam * 100

print()
print("=== SENARYO Y v3 (ASIMETRIK ARACCINSI) ===")
print(f"Toplam atama: {len(senaryo_y_v3)}")
print(f"  Metrobus eslesme: {len(metrobus_eslesme)}")
print(f"  Otobus eslesme: {len(otobus_eslesme)}")
print(f"Mevcut toplam:                  {mevcut_toplam:,.0f}")
print(f"Onerilen (Y v3, asimetrik):     {onerilen_y_v3:,.0f}")
print(f"KAZANC: %{kazanc_y_v3:.2f}")

kazanc_y = kazanc_y_v3
onerilen_y_toplam = onerilen_y_v3


Hat ARACCINSI gereksinim dagilimi:
ARACCINSI_gerek
SERBEST    720
KORUKLU      8
Name: count, dtype: int64

METROBUS hat sayisi: 8
KORUKLU arac toplam: 1278
Metrobus icin tahsis edilen guvenilir KORUKLU: 8
Otobus hatlarina kalan KORUKLU: 1270

Otobus havuzu (KORUKLU_kalan + SOLO): 3500 arac
Otobus hat sayisi: 720

=== SENARYO Y v3 (ASIMETRIK ARACCINSI) ===
Toplam atama: 3508
  Metrobus eslesme: 8
  Otobus eslesme: 3500
Mevcut toplam:                  61,073
Onerilen (Y v3, asimetrik):     48,013
KAZANC: %21.38


---
## 12. H4 Testi: Senaryo Kazanc Karsilastirma

Iki senaryo ne kadar kazanç sağlıyor? Hangi gerçekçi?


In [13]:
# BOLUM 12: H4 - Kazanc karsilastirma
print('=== H4 TEST: ANOMALI DUZELTME KAZANCI ===')
print(f'\n{"Senaryo":30s} {"Toplam Skor":>15s} {"Kazanc":>10s}')
print('-'*60)
print(f'{"Mevcut atama":30s} {mevcut_toplam:>15,.0f} {"(referans)":>10s}')
print(f'{"X (Garaj SABIT)":30s} {onerilen_toplam:>15,.0f} {kazanc:>9.2f}%')
print(f'{"Y (Garaj SERBEST - teorik)":30s} {onerilen_y_toplam:>15,.0f} {kazanc_y:>9.2f}%')

print(f'\n=== H4 KARAR ===')
if kazanc > 5:
    print(f'H4 PASSED: Garaj sabit senaryo %{kazanc:.1f} kazanc - operasyonel anlamli')
elif kazanc > 0:
    print(f'H4 KISMI: Kazanc var ama %{kazanc:.1f} kucuk - mevcut atama zaten yaklasik optimal')
else:
    print(f'H4 FAILED: Onerilen senaryo daha kotu - mevcut atama optimaldi')

print(f'\n=== YORUM ===')
print(f'Gerçekci tavan: %{kazanc:.2f} (garaj kısıtlı - hayata gecebilir)')
print(f'Teorik tavan:   %{kazanc_y:.2f} (garaj serbest - veri-bazli, operasyonel zor)')
fark = kazanc_y - kazanc
print(f'Operasyonel kisitin maliyeti: %{fark:.2f} potansiyel kazanc kaybi')


=== H4 TEST: ANOMALI DUZELTME KAZANCI ===

Senaryo                            Toplam Skor     Kazanc
------------------------------------------------------------
Mevcut atama                            61,073 (referans)
X (Garaj SABIT)                         60,716      0.58%
Y (Garaj SERBEST - teorik)              48,013     21.38%

=== H4 KARAR ===
H4 KISMI: Kazanc var ama %0.6 kucuk - mevcut atama zaten yaklasik optimal

=== YORUM ===
Gerçekci tavan: %0.58 (garaj kısıtlı - hayata gecebilir)
Teorik tavan:   %21.38 (garaj serbest - veri-bazli, operasyonel zor)
Operasyonel kisitin maliyeti: %20.80 potansiyel kazanc kaybi


---
## 13. FEATURES_FINAL Hazirligi - Tum Dogrulanmis Feature'lar

Onceki 8 analiz + bu analizden cikan tum ML feature'lar.


In [14]:
# BOLUM 13: FEATURES_FINAL
features_final = pd.DataFrame([
    {"Feature":"cascade_risk_skor", "Analiz":"A1 Cascade", "r":None, "Leakage":"Test edilecek", "Karar":"V6 aday"},
    {"Feature":"sistem_cas_lift", "Analiz":"A1 Cascade", "r":None, "Leakage":"Test edilecek", "Karar":"V6 aday"},
    {"Feature":"sofor_glob_skor", "Analiz":"A2 Sofor", "r":+0.391, "Leakage":"%81 dusus", "Karar":"ATLA (leakage)"},
    {"Feature":"farkli_arac_sayisi", "Analiz":"A2 Sofor", "r":None, "Leakage":"Test edilecek", "Karar":"V6 aday"},
    {"Feature":"egim_maruziyet", "Analiz":"A3 Topografya", "r":+0.135, "Leakage":"Stable", "Karar":"V6 EKLE"},
    {"Feature":"son_kaza_gun", "Analiz":"A4 Kaza", "r":+0.13, "Leakage":"Stable", "Karar":"V6 EKLE"},
    {"Feature":"garaj_sistem_lift", "Analiz":"A5 Garaj", "r":+0.213, "Leakage":"%3 dusus (stable)", "Karar":"V6 EKLE"},
    {"Feature":"garaj_ort_skor", "Analiz":"A5 Garaj", "r":+0.132, "Leakage":"Stable", "Karar":"V6 EKLE"},
    {"Feature":"garaj_ciddi_oran", "Analiz":"A5 Garaj", "r":+0.100, "Leakage":"Stable", "Karar":"V6 EKLE"},
    {"Feature":"garaj_marka_lift", "Analiz":"A5 Garaj", "r":+0.084, "Leakage":"Stable", "Karar":"V6 EKLE"},
    {"Feature":"gunluk_sefer_sayisi", "Analiz":"A6 Yorgunluk", "r":-0.267, "Leakage":"%93 dusus", "Karar":"ATLA (leakage + ters)"},
    {"Feature":"son_30g_top", "Analiz":"A6 Yorgunluk", "r":-0.014, "Leakage":"%56 dusus", "Karar":"ATLA"},
    {"Feature":"yakit_turu_cng", "Analiz":"A7 Yakit", "r":+0.036, "Leakage":"M4 k=-0.358", "Karar":"V6 EKLE (kategorik)"},
    {"Feature":"verimsizlik_skoru", "Analiz":"A7 Yakit", "r":+0.220, "Leakage":"Stable", "Karar":"V6 EKLE"},
    {"Feature":"kritiklik_skoru", "Analiz":"A8 Guvenlik", "r":+0.479, "Leakage":"Stability 0.15 ZAYIF", "Karar":"OPERASYONEL ONLY"},
    {"Feature":"hat_zorluk", "Analiz":"A9 Atama", "r":None, "Leakage":"Hat sabit, leakage yok", "Karar":"V6 EKLE"},
    {"Feature":"anomali_skor", "Analiz":"A9 Atama", "r":None, "Leakage":"Test edilecek", "Karar":"V6 aday"},
])
print("=== FEATURES_FINAL TABLOSU ===")
print(features_final.to_string(index=False))

print()
print("=== OZET ===")
n_ekle = features_final["Karar"].str.startswith("V6 EKLE").sum()
n_aday = (features_final["Karar"]=="V6 aday").sum()
n_atla = features_final["Karar"].str.contains("ATLA").sum()
n_op = features_final["Karar"].str.contains("OPERASYONEL").sum()
print(f"V6 EKLE: {n_ekle} feature (kategorik dahil)")
print(f"V6 aday: {n_aday} feature (leakage testi gerek)")
print(f"ATLA:    {n_atla} feature (leakage veya ters yon)")
print(f"OPERASYONEL ONLY: {n_op} feature")
print(f"TOPLAM:  {n_ekle + n_aday + n_atla + n_op}")


=== FEATURES_FINAL TABLOSU ===
            Feature        Analiz      r                Leakage                 Karar
  cascade_risk_skor    A1 Cascade    NaN          Test edilecek               V6 aday
    sistem_cas_lift    A1 Cascade    NaN          Test edilecek               V6 aday
    sofor_glob_skor      A2 Sofor  0.391              %81 dusus        ATLA (leakage)
 farkli_arac_sayisi      A2 Sofor    NaN          Test edilecek               V6 aday
     egim_maruziyet A3 Topografya  0.135                 Stable               V6 EKLE
       son_kaza_gun       A4 Kaza  0.130                 Stable               V6 EKLE
  garaj_sistem_lift      A5 Garaj  0.213      %3 dusus (stable)               V6 EKLE
     garaj_ort_skor      A5 Garaj  0.132                 Stable               V6 EKLE
   garaj_ciddi_oran      A5 Garaj  0.100                 Stable               V6 EKLE
   garaj_marka_lift      A5 Garaj  0.084                 Stable               V6 EKLE
gunluk_sefer_sayisi  A6

---
## 14. Operasyonel Cikti CSV + iett_panel Tasarim Onerisi

Top 50 anomali listesini operasyonel ekibe vermek icin CSV.


In [15]:
# BOLUM 14: Operasyonel cikti - Multi-Label Aksiyon Onerileri (B2)
# Her arac ihtiyac duydugu HER aksiyonu alir (paralel, sirali if-elif degil).
# Esikler: B kategorisi onayli (2026-05-20).

output_csv = top50_anomali.copy()

def aksiyon_oneri_multi(r):
    """Her arac icin uygun aksiyonlarin listesi (virgulle birlestirilmis).
    Esikler arasinda cakisma normal — arac birden cok aksiyon alabilir."""
    yas = r["yas"]
    skor = r["arac_kritiklik"]
    hat_zor = r["hat_zorluk"]
    garaj = r["GARAJ"]

    aksiyonlar = []

    # 1) MOTOR DENETIMI: yasli + yuksek risk
    if yas >= 15 and skor > 70:
        aksiyonlar.append("MOTOR_DENETIMI")

    # 2) BAKIM ONCELIGI: zor hat + orta-ustu riskli arac
    if hat_zor > 40 and skor > 60:
        aksiyonlar.append("BAKIM_ONCELIGI")

    # 3) GARAJ ROTASYONU: yipratici garajda yeni arac (A5 Bolum 18)
    if garaj in ["Edirnekapı", "Hasanpaşa"] and yas < 5:
        aksiyonlar.append("GARAJ_ROTASYONU")

    # 4) YENILEME ADAYI: cok yasli + yuksek kritiklik (Edirnekapi Capacity ornegi)
    if yas >= 17 and skor > 75:
        aksiyonlar.append("YENILEME_ADAYI")

    # Hicbiri tutmazsa IZLEME
    if not aksiyonlar:
        aksiyonlar.append("IZLEME")

    return " + ".join(aksiyonlar)

def aksiyon_aciklama(aksiyonlar_str):
    """Kisa aciklama metni"""
    mapping = {
        "MOTOR_DENETIMI": "yasli + yuksek risk",
        "BAKIM_ONCELIGI": "zor hat + riskli arac",
        "GARAJ_ROTASYONU": "yeni arac yipratici garajda",
        "YENILEME_ADAYI": "filo yenileme onerisi",
        "IZLEME": "rutin izleme",
    }
    return "; ".join(mapping.get(a, a) for a in aksiyonlar_str.split(" + "))

output_csv["aksiyon_oneri"] = output_csv.apply(aksiyon_oneri_multi, axis=1)
output_csv["aksiyon_aciklama"] = output_csv["aksiyon_oneri"].apply(aksiyon_aciklama)

# CSV export (panel/V6.5 etkilemez, sadece datathon raporu)
output_csv.to_csv("analiz9_anomali_listesi.csv", index=False)
print(f"Operasyonel CSV export: analiz9_anomali_listesi.csv ({len(output_csv)} satir)")

# Tek tek aksiyon sayilari (multi-label oldugu icin satir sayisindan fazla olabilir)
print()
print("=== TEKIL AKSIYON SAYILARI (multi-label) ===")
tum_aksiyonlar = []
for a_str in output_csv["aksiyon_oneri"]:
    tum_aksiyonlar.extend(a_str.split(" + "))
import collections
aksiyon_sayilari = collections.Counter(tum_aksiyonlar)
for ak, n in aksiyon_sayilari.most_common():
    print(f"  {ak:20s}: {n:3d} arac")

print()
print("=== ARAC BASINA AKSIYON SAYISI DAGILIMI ===")
output_csv["n_aksiyon"] = output_csv["aksiyon_oneri"].apply(lambda s: len(s.split(" + ")))
print(output_csv["n_aksiyon"].value_counts().sort_index().to_string())

print()
print("=== AKSIYON KOMBINASYON DAGILIMI ===")
print(output_csv["aksiyon_oneri"].value_counts().to_string())

print()
print("=== TOP 10 OPERASYONEL ONERI ===")
print(output_csv[["KAPINO","GARAJ","yas","arac_kritiklik","ana_HAT","hat_zorluk","aksiyon_oneri"]].head(10).to_string(index=False))

print()
print("=== iett_panel ONERI (multi-label rozetler) ===")
print("Yeni Sekme: Akilli Atama Uyarisi")
print("  - Top 50 anomali listesi + COKLU rozet sistemi (arac birden fazla aksiyon alabilir)")
print("  - Rozet renkleri:")
print("    [KIRMIZI] MOTOR DENETIMI")
print("    [TURUNCU] BAKIM ONCELIGI")
print("    [SARI]    GARAJ ROTASYONU")
print("    [MOR]     YENILEME ADAYI")
print("    [YESIL]   IZLEME")
print("  - Kabul Et / Erteler / Reddet butonlari (operasyonel ekip geri bildirim)")
print("  - Hat zorluk x Arac kritiklik scatter plot (anomalileri highlight)")


Operasyonel CSV export: analiz9_anomali_listesi.csv (50 satir)

=== TEKIL AKSIYON SAYILARI (multi-label) ===
  MOTOR_DENETIMI      :  50 arac
  BAKIM_ONCELIGI      :  50 arac
  YENILEME_ADAYI      :   7 arac

=== ARAC BASINA AKSIYON SAYISI DAGILIMI ===
n_aksiyon
2    43
3     7

=== AKSIYON KOMBINASYON DAGILIMI ===
aksiyon_oneri
MOTOR_DENETIMI + BAKIM_ONCELIGI                     43
MOTOR_DENETIMI + BAKIM_ONCELIGI + YENILEME_ADAYI     7

=== TOP 10 OPERASYONEL ONERI ===
KAPINO      GARAJ  yas  arac_kritiklik ana_HAT  hat_zorluk                                    aksiyon_oneri
 M4003 Edirnekapı 18.0            78.8    34BZ        49.4 MOTOR_DENETIMI + BAKIM_ONCELIGI + YENILEME_ADAYI
 M3292 Edirnekapı 16.0            78.0    34BZ        49.4                  MOTOR_DENETIMI + BAKIM_ONCELIGI
 M4544 Edirnekapı 17.0            76.9    34BZ        49.4 MOTOR_DENETIMI + BAKIM_ONCELIGI + YENILEME_ADAYI
 M6576 Edirnekapı 18.0            76.3    34BZ        49.4 MOTOR_DENETIMI + BAKIM_ONCELIGI + 

---
## 15. Sonuc Konsolidasyonu - Tum Hipotez Testleri


In [16]:
# BOLUM 15: Hipotez test ozeti
print("="*70)
print("ANALIZ 9 - HIPOTEZ TEST OZETI")
print("="*70)
print()
print(f"H1: Mevcut atama optimal degil (hat_zorluk × arac_kritiklik korelasyon)")
print(f"    Bulgu: r = {r_h1:+.4f}, p = {p_h1:.4f}")
print(f"    Karar: {'PASS' if abs(r_h1) > 0.05 else 'NEUTRAL'}")
print()
print(f"H2: ARACCINSI uyumsuzluk nadir (METROBUS ozelinde)")
print(f"    Bulgu: Metrobus uyumsuzluk %{metrobus_uyumsuzluk_pct:.4f}")
print(f"    Karar: {'PASS' if metrobus_uyumsuzluk_pct < 0.5 else 'FAIL'}")
print()
print(f"H3: Yasli garajlar zor hatlarda (A3 dogrulama)")
print(f"    Bulgu: garaj yas × hat_zorluk r = {r_h3:+.4f}")
print(f"    Karar: {'PASS' if r_h3 > 0.3 else 'FAIL'}")
print()
print(f"H4: Anomali duzeltme kazanc saglar")
print(f"    Garaj SABIT: %{kazanc:.2f}, Garaj SERBEST: %{kazanc_y:.2f}")
print(f"    Karar: {'PASS' if kazanc > 5 else 'KISMI' if kazanc > 0 else 'FAIL'}")


ANALIZ 9 - HIPOTEZ TEST OZETI

H1: Mevcut atama optimal degil (hat_zorluk × arac_kritiklik korelasyon)
    Bulgu: r = +0.3384, p = 0.0000
    Karar: PASS

H2: ARACCINSI uyumsuzluk nadir (METROBUS ozelinde)
    Bulgu: Metrobus uyumsuzluk %0.0047
    Karar: PASS

H3: Yasli garajlar zor hatlarda (A3 dogrulama)
    Bulgu: garaj yas × hat_zorluk r = +0.4710
    Karar: PASS

H4: Anomali duzeltme kazanc saglar
    Garaj SABIT: %0.58, Garaj SERBEST: %21.38
    Karar: KISMI


---
## 16. Kisitlamalar ve ML V6 Baglantisi


In [17]:
# BOLUM 16: Kisitlamalar
print("=== KISITLAMALAR ===")
print()
kisitlamalar = [
    "1. Veri 6 ay - yillik mevsimsellik yansitilmadi",
    "2. ARACTIPI ayrımı yapilmadi (proje karari) - sadece ML feature olarak ele alindi",
    "3. yolcu_doluluk guvenilmez (proje karari)",
    "4. Atama olasiligi: arac × hat eslesmesi sefer bazli, saat bazli degil",
    "5. Operatorun mevcut atama politikasinin tarihsel nedenleri olabilir",
    "6. Greedy algoritma optimal degil (gercek optimization Hungarian/LP)",
    "7. Motor yenileme verisi yok (yas feature zayif, A8 ile tutarli)",
    "8. anomali_skor stabilite testi yapilmadi (gelecek is)",
]
for k in kisitlamalar:
    print(k)

print()
print("=== ML V6 BAGLANTISI ===")
print("FEATURES_FINAL tablosu (Bolum 13) ile V6 hazır:")
print(f"  - {n_ekle} feature V6 EKLENECEK (kategorik dahil)")
print(f"  - {n_aday} feature aday (leakage testinden gecmesi gerek)")
print(f"  - {n_atla} feature ATLANACAK (leakage veya ters yon)")
print(f"  - {n_op} feature OPERASYONEL ONLY (kritiklik_skoru)")

print()
print("=== SONRAKI ADIMLAR ===")
print("1. ANALIZ_9_SONUCLAR.md yazimi")
print("2. FEATURES_FINAL.md hazirlanmasi (Bolum 13 tablosundan)")
print("3. ML_MODEL_V6 - feature engineering + AUC karsilastir (V5: 0.762)")
print("4. iett_panel entegrasyonu (Yuksek Risk + Akilli Atama sekmeleri)")


=== KISITLAMALAR ===

1. Veri 6 ay - yillik mevsimsellik yansitilmadi
2. ARACTIPI ayrımı yapilmadi (proje karari) - sadece ML feature olarak ele alindi
3. yolcu_doluluk guvenilmez (proje karari)
4. Atama olasiligi: arac × hat eslesmesi sefer bazli, saat bazli degil
5. Operatorun mevcut atama politikasinin tarihsel nedenleri olabilir
6. Greedy algoritma optimal degil (gercek optimization Hungarian/LP)
7. Motor yenileme verisi yok (yas feature zayif, A8 ile tutarli)
8. anomali_skor stabilite testi yapilmadi (gelecek is)

=== ML V6 BAGLANTISI ===
FEATURES_FINAL tablosu (Bolum 13) ile V6 hazır:
  - 9 feature V6 EKLENECEK (kategorik dahil)
  - 4 feature aday (leakage testinden gecmesi gerek)
  - 3 feature ATLANACAK (leakage veya ters yon)
  - 1 feature OPERASYONEL ONLY (kritiklik_skoru)

=== SONRAKI ADIMLAR ===
1. ANALIZ_9_SONUCLAR.md yazimi
2. FEATURES_FINAL.md hazirlanmasi (Bolum 13 tablosundan)
3. ML_MODEL_V6 - feature engineering + AUC karsilastir (V5: 0.762)
4. iett_panel entegrasyonu 